# <span style="font-width:bold; font-size: 3rem; color:#1EB182;"><img src="../images/icon102.png" width="38px"></img> **Hopsworks Feature Store** </span><span style="font-width:bold; font-size: 3rem; color:#333;">- Part 02: Training Pipeline</span>


<span style="font-width:bold; font-size: 1.4rem;">This notebook explains how to read from a feature group, create training dataset within the feature store, train a model and save it to model registry.</span>

## 🗒️ This notebook is divided into the following sections:

1. Fetch Feature Groups.
2. Define Transformation functions.
3. Create Feature Views.
4. Create Training Dataset with training, validation and test splits.
5. Train the model.
6. Register model in Hopsworks Model Registry.
7. Create the Deployment.

![part2](../images/02_training-dataset.png) 

## <span style='color:#ff5f27'> 📝 Imports

In [ ]:
!pip install -U xgboost --quiet

In [ ]:
import joblib
import os
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

import xgboost as xgb
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

from features import embeddings

# Mute warnings
import warnings
warnings.filterwarnings("ignore")

## <span style="color:#ff5f27;"> 📡 Connecting to Hopsworks Feature Store </span>

In [ ]:
import hopsworks

project = hopsworks.login()

fs = project.get_feature_store()

---

## <span style="color:#ff5f27;"> 🔪 Feature Selection </span>

You will start by selecting all the features you want to include for model training/inference.

In [ ]:
# Retrieve feature groups.
trans_fg = fs.get_feature_group(
    name='transactions_fraud_online_fg', 
    version=1,
)
profile_online_fg = fs.get_feature_group(
    name='profile_fraud_online_fg', 
    version=1,
)
payer_emb_fg = fs.get_feature_group(
    name='payer_embeddings_fraud_online_fg',
    version=1,
)

# Select features for training dataset
selected_features = trans_fg.select_all() \
    .join(profile_online_fg.select_all(include_primary_key=False)) \
    .join(payer_emb_fg.select(["payer_behavior_embedding", "payer_transaction_sequence_embedding"]))

In [ ]:
# Uncomment this if you would like to view your selected features
# selected_features.show(5)

Recall that you computed the features in `transactions_fraud_online_fg`. If you had created multiple feature groups with identical schema for different window lengths, and wanted to include them in the join you would need to include a prefix argument in the join to avoid feature name clash. See the [documentation](https://docs.hopsworks.ai/feature-store-api/latest/generated/api/query_api/#join) for more details.

---

### <span style="color:#ff5f27;"> 🤖 Transformation Functions </span>


You will preprocess our data using *min-max scaling* on numerical features and *label encoding* on categorical features. To do this you simply define a mapping between our features and transformation functions. This ensures that transformation functions such as *min-max scaling* are fitted only on the training data (and not the validation/test data), which ensures that there is no data leakage.

In [ ]:
# Import transformation functions from Hopsworks.
from hopsworks.hsfs.builtin_transformations import label_encoder

# Map features to transformation functions.
transformation_functions = [
    label_encoder("country"),
    label_encoder("gender"),
]

## <span style="color:#ff5f27;"> ⚙️ Feature View Creation </span>

The Feature Views allows schema in form of a query with filters, define a model target feature/label and additional transformation functions.
In order to create or get a Feature View you may use `fs.get_or_create_feature_view()`

In [ ]:
# Get or create the 'transactions_fraud_online_fv' feature view
feature_view = fs.get_or_create_feature_view(
    name='transactions_fraud_online_fv',
    version=1,
    query=selected_features,
    labels=["fraud_label"],
    transformation_functions=transformation_functions,
    logging_enabled=True
)

## <span style="color:#ff5f27;"> 🏋️ Training Dataset </span>

In [ ]:
# Training/Test splits, datasets creation. Using timerange arguments.
train_start = "2022/01/01"
train_end = "2022/03/10"
test_start = "2022/03/10"
test_end = "2022/03/31"

X_train, X_test, y_train, y_test = feature_view.train_test_split(
    train_start=train_start,
    train_end=train_end,
    test_start=test_start,
    test_end=test_end,
)

The feature view and training dataset are now visible in the UI

![fg-overview](../images/fv_overview.gif)

In [ ]:
# Sort the X_train by 'datetime'
X_train = X_train.sort_values("datetime")

# Reindex the y_train based on the sorted index of X_train
y_train = y_train.reindex(X_train.index)

In [ ]:
# Sort the X_test DataFrame by 'datetime'
X_test = X_test.sort_values("datetime")

# Reindex the y_test based on the sorted index of X_test
y_test = y_test.reindex(X_test.index)

In [ ]:
# Extract the credit card number of the first sample from the test features (X_test) DataFrame
test_sample = X_test.cc_num.values[0]

In [ ]:
# Drop identifier columns not used by the model
X_train = X_train.drop(["tid", "cc_num", "datetime"], axis=1)
X_test = X_test.drop(["tid", "cc_num", "datetime"], axis=1)

# Flatten the embedding (array) features into per-dimension scalar columns for XGBoost
X_train = embeddings.flatten_embedding_features(X_train)
X_test = embeddings.flatten_embedding_features(X_test)

In [ ]:
# Display the normalized value counts of the training labels (y_train)
y_train.value_counts(normalize=True)

Notice that the distribution is extremely skewed, which is natural considering that fraudulent transactions make up a tiny part of all transactions. Thus you should somehow address the class imbalance. There are many approaches for this, such as weighting the loss function, over- or undersampling, creating synthetic data, or modifying the decision threshold. In this example, you will use the simplest method which is to just supply a class weight parameter to our learning algorithm. The class weight will affect how much importance is attached to each class, which in our case means that higher importance will be placed on positive (fraudulent) samples.

---

## <span style="color:#ff5f27;"> 🧬 Modeling</span>

Next you will train a model. Here, you set larger class weight for the positive class.

In [ ]:
# Initialize an XGBoost classifier
model = xgb.XGBClassifier()

# Train the classifier using the training features (X_train) and labels (y_train)
model.fit(X_train, y_train)

In [ ]:
# Predict the training set
y_pred_train = model.predict(X_train)

# Predict the test set
y_pred_test = model.predict(X_test)

In [ ]:
y_pred_test

In [ ]:
X_test.head(3)

In [ ]:
# Compute f1 score
metrics = {
    "f1_score": f1_score(y_test, y_pred_test, average='macro')
}
metrics

In [ ]:
# Calculate the confusion matrix for the test set predictions
results = confusion_matrix(
    y_test, 
    y_pred_test, 
    labels=[False, True],
)

# Print the confusion matrix
print(results)

---

## <span style="color:#ff5f27;">📝 Register model</span>

One of the features in Hopsworks is the model registry. This is where we can store different versions of models and compare their performance. Models from the registry can then be served as API endpoints.

In [ ]:
# Specify the model directory
model_dir = "fraud_online_model"
images_dir = os.path.join(model_dir, "images")

# Create directories if they don't exist
os.makedirs(images_dir, exist_ok=True)

In [ ]:
# Save the trained XGBoost model
joblib.dump(model, os.path.join(model_dir, "xgboost_fraud_online_model.pkl"))

In [ ]:
# Create a DataFrame from the confusion matrix results
df_cm = pd.DataFrame(
    results, 
    ['True Normal', 'True Fraud'],
    ['Pred Normal', 'Pred Fraud']
)

# Create and save the confusion matrix heatmap
plt.figure(figsize=(8, 6))
cm = sns.heatmap(
    df_cm, 
    annot=True,
    fmt='d',                 # Use integer format for numbers
    cmap='RdPu',             # Use a color palette that works well for binary classification
    annot_kws={'size': 12},  # Increase annotation text size
    cbar=True                # Include color bar
)

# Add title and labels
plt.title('Confusion Matrix for Fraud Detection')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

# Adjust layout and save
plt.tight_layout()
plt.savefig(os.path.join(images_dir, "confusion_matrix.png"), dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# Get the model registry
mr = project.get_model_registry()

# Create a Python model in the model registry
fraud_model = mr.python.create_model(
    name="xgboost_fraud_online_model", 
    description="Fraud Online Predictor", # Add a description for the model
    metrics=metrics,                      # Specify the metrics used to evaluate the model
    input_example=[4467360740682089],     # Example input for testing deployments
    feature_view=feature_view,            # Add a feature view to the model
)

# Save the model to the specified model directory
fraud_model.save(model_dir)

---

## <a class="anchor" id="1.5_bullet" style="color:#ff5f27"> 🚀 Model Deployment</a>


### About Model Serving
Models can be served via KFServing or "default" serving, which means a Docker container exposing a Flask server. For KFServing models, or models written in Tensorflow, you do not need to write a prediction file (see the section below). However, for sklearn models using default serving, you do need to proceed to write a prediction file.

In order to use KFServing, you must have Kubernetes installed and enabled on your cluster.

### <span style="color:#ff5f27;">📎 Predictor script for Python models</span>


Scikit-learn and XGBoost models are deployed as Python models, in which case you need to provide a **Predict** class that implements the **predict** method. The **predict()** method invokes the model on the inputs and returns the prediction as a list.

The **init()** method is run when the predictor is loaded into memory, loading the model from the local directory it is materialized to, *ARTIFACT_FILES_PATH*.

The directive "%%writefile" writes out the cell before to the given Python file. We will use the **predict_example.py** file to create a deployment for our model. 

In [ ]:
%%writefile predict_example.py
import os
import numpy as np
import hopsworks
import joblib


class Predict(object):

    def __init__(self, async_logger, model):
        """ Initializes the serving state, reads a trained model"""        
        # Get feature store handle
        project = hopsworks.login()
        self.mr = project.get_model_registry()

        # Retrieve the feature view from the model
        retrieved_model = self.mr.get_model(
            name="xgboost_fraud_online_model",
            version=1,
        )
        self.feature_view = retrieved_model.get_feature_view()
        
        # Initialize serving and async feature logging
        self.feature_view.init_serving(feature_logger=async_logger)

        # Load the trained model
        self.hopsworks_model = model

        self.model = joblib.load(os.environ["MODEL_FILES_PATH"] + "/xgboost_fraud_online_model.pkl")

        print("Initialization Complete")

    def predict(self, inputs):
        """ Serves a prediction request usign a trained model"""
        untransformed_feature_vector = self.feature_view.get_feature_vector({"cc_num": inputs[0][0]}, transform=False)
        feature_vector = self.feature_view.transform(untransformed_feature_vector)
        # Drop the identifier columns (tid, datetime, cc_num) and flatten any embedding
        # (array-valued) feature into scalars, matching the training-time layout.
        indexes_to_remove = [0, 1, 2]
        model_input = []
        for j, value in enumerate(feature_vector):
            if j in indexes_to_remove:
                continue
            if isinstance(value, (list, np.ndarray)):
                model_input.extend(np.asarray(value, dtype=np.float64).ravel().tolist())
            else:
                model_input.append(value)
        prediction = self.model.predict(np.asarray(model_input).reshape(1, -1)).tolist() # Numpy Arrays are not JSON serializable
        self.feature_view.log(untransformed_features=[untransformed_feature_vector],
            transformed_features=[feature_vector],
            predictions=[prediction],
            model=self.hopsworks_model)
        return prediction

If you wonder why we use the path Models/fraud_tutorial_model/1/model.pkl, it is useful to know that the Data Sets tab in the Hopsworks UI lets you browse among the different files in the project. Registered models will be found underneath the Models directory. Since you saved you model with the name fraud_tutorial_model, that's the directory you should look in. 1 is just the version of the model you want to deploy.

This script needs to be put into a known location in the Hopsworks file system. Let's call the file predict_example.py and put it in the Models directory.

In [ ]:
# `predict_example.py` is uploaded to HopsFS automatically by the SDK
# when the deployment is saved; no manual dataset_api.upload step needed.

### Create the deployment
Here, you fetch the model you want from the model registry and define a configuration for the deployment. For the configuration, you need to specify the serving type (default or KFserving).

In [ ]:
# Deploy the fraud model
deployment = fraud_model.deploy(
    name="fraudonlinemodeldeployment",  # Specify a name for the deployment
    script_file="predict_example.py",  # Local path; SDK auto-uploads on save
)

In [ ]:
# Print the name of the deployment
print("Deployment: " + deployment.name)

# Display information about the deployment
deployment.describe()

In [ ]:
print("Deployment is warming up...")
time.sleep(45)

#### The deployment has now been registered. However, to start it you need to run the following command:

In [ ]:
# Start the deployment and wait for it to be in a running state for up to 300 seconds
deployment.start(await_running=300)

In [ ]:
# Get the current state of the deployment
deployment.get_state().describe()

---
## <span style="color:#ff5f27;"> 🔁 Model Monitoring & Drift-Triggered Re-training </span>

With the model deployed and logging its inputs, you can configure **feature monitoring** on the served model. Hopsworks periodically compares a *detection window* (recent inference data) against a *reference* (the model's training dataset) and flags a **shift** when a metric crosses a threshold.

For the embedding features you use the embedding-specific metric `centroid_distance` (the L2 distance between the detection and reference embedding centroids). For the scalar `amount` feature you use the `PSI` distribution metric.

On the `payer_transaction_sequence_embedding` config you also enable **automated re-training**: after `retrain_model_after_num_shifts` consecutive shifts, Hopsworks runs a re-training job. A notebook-trained model has no originating job, so you first write the training program to a script, register it as a Hopsworks job, and pass it explicitly as `model_retraining_job`. The embeddings are already stored as features, so the re-training job reads them straight from the training dataset and does not re-run the SentenceTransformer.

In [ ]:
%%writefile training_job.py
import os
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import f1_score
import hopsworks


def flatten_embedding_features(df):
    """Expand list/array columns into per-dimension scalar columns, preserving order."""
    pieces = []
    for col in df.columns:
        sample = df[col].iloc[0]
        if isinstance(sample, (list, np.ndarray)):
            mat = np.vstack(df[col].apply(lambda v: np.asarray(v, dtype=np.float64)).values)
            pieces.append(pd.DataFrame(
                mat,
                columns=[f"{col}_{i}" for i in range(mat.shape[1])],
                index=df.index,
            ))
        else:
            pieces.append(df[[col]])
    return pd.concat(pieces, axis=1)


# Connect and fetch the feature view (embeddings are already stored, no re-encoding needed)
project = hopsworks.login()
fs = project.get_feature_store()
mr = project.get_model_registry()

feature_view = fs.get_feature_view("transactions_fraud_online_fv", version=1)

X_train, X_test, y_train, y_test = feature_view.train_test_split(
    train_start="2022/01/01",
    train_end="2022/03/10",
    test_start="2022/03/10",
    test_end="2022/03/31",
)

X_train = X_train.sort_values("datetime")
y_train = y_train.reindex(X_train.index)
X_test = X_test.sort_values("datetime")
y_test = y_test.reindex(X_test.index)

X_train = X_train.drop(["tid", "cc_num", "datetime"], axis=1)
X_test = X_test.drop(["tid", "cc_num", "datetime"], axis=1)

X_train = flatten_embedding_features(X_train)
X_test = flatten_embedding_features(X_test)

model = xgb.XGBClassifier()
model.fit(X_train, y_train)
metrics = {"f1_score": f1_score(y_test, model.predict(X_test), average="macro")}

model_dir = "fraud_online_model"
os.makedirs(model_dir, exist_ok=True)
joblib.dump(model, os.path.join(model_dir, "xgboost_fraud_online_model.pkl"))

fraud_model = mr.python.create_model(
    name="xgboost_fraud_online_model",
    description="Fraud Online Predictor (retrained)",
    metrics=metrics,
    input_example=[4467360740682089],
    feature_view=feature_view,
)
fraud_model.save(model_dir)
print("Re-training complete; registered a new model version.")

In [ ]:
# Upload the re-training program and register it as a Python job
dataset_api = project.get_dataset_api()
dataset_api.upload("training_job.py", "Resources", overwrite=True)

job_api = project.get_job_api()
job_config = job_api.get_configuration("PYTHON")
job_config["appPath"] = "/Resources/training_job.py"
retrain_job = job_api.create_job("fraud_online_retrain", job_config)

Now create the monitoring configurations. Each call to `create_model_monitoring(...)` is followed by a fluent chain that sets the **detection window** (recent inference data), the **reference** (the model's training dataset), and the **comparison metric**. The sequence-embedding config additionally enables drift-triggered re-training via `retrain_model_after_num_shifts` and the job registered above.

In [ ]:
# Monitor the scalar transaction amount for distribution drift (PSI)
amount_monitoring = feature_view.create_model_monitoring(
    name="amount_drift",
    model_name="xgboost_fraud_online_model",
    model_version=1,
)
amount_monitoring.with_detection_window(time_offset="1d") \
    .with_reference_training_dataset() \
    .compare_on_distribution(feature_name="amount", metric="PSI", threshold=0.2) \
    .save()

# Monitor the recent-sequence embedding for centroid drift, and enable automated
# re-training after 3 consecutive shifts using the registered job
sequence_monitoring = feature_view.create_model_monitoring(
    name="sequence_embedding_drift",
    model_name="xgboost_fraud_online_model",
    model_version=1,
    retrain_model_after_num_shifts=3,
    model_retraining_job=retrain_job,
)
sequence_monitoring.with_detection_window(time_offset="1d") \
    .with_reference_training_dataset() \
    .compare_on(
        feature_name="payer_transaction_sequence_embedding",
        metric="centroid_distance",
        threshold=0.1,
    ) \
    .save()

# Monitor the long-term behavior embedding for centroid drift (population shift)
behavior_monitoring = feature_view.create_model_monitoring(
    name="behavior_embedding_drift",
    model_name="xgboost_fraud_online_model",
    model_version=1,
)
behavior_monitoring.with_detection_window(time_offset="1d") \
    .with_reference_training_dataset() \
    .compare_on(
        feature_name="payer_behavior_embedding",
        metric="centroid_distance",
        threshold=0.1,
    ) \
    .save()

In [ ]:
# To troubleshoot you can use `get_logs()` method
deployment.get_logs(component='predictor')

### Stop Deployment
To stop the deployment you simply run:

In [ ]:
# Stop the deployment and wait for it to be in a stopped state for up to 180 seconds
deployment.stop(await_stopped=180)

---
## <span style="color:#ff5f27;">⏭️ **Next:** Part 03: Inference Pipeline</span>

In the following notebook you will use your model for Serving Vector Inference.
